# Лаба 6



```
Метод окаймления для решения слау
```



In [ ]:
import numpy as np

def solve_by_bordering_with_det(A, b, tol=1e-15, verbose=True):
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)

    n = A.shape[0]

    if abs(A[0, 0]) < tol:
        raise ValueError("A[0,0] is zero — leading principal minor is singular")

    A_inv = np.array([[1.0 / A[0, 0]]])
    x = np.array([b[0] / A[0, 0]])
    det = A[0, 0]

    if verbose:
        print("Шаг 1 (k=1): инициализация")
        print(f"  A_1 = [{A[0,0]:.4f}]")
        print(f"  A_1⁻¹ = [{1.0 / A[0,0]:.6f}]")
        print(f"  x = [{x[0]:.6f}]")
        print(f"  det = {det:.6f}")

    for k in range(1, n):
        u = A[:k, k]
        v = A[k, :k]
        a_kk = A[k, k]
        b_k1 = b[k]

        s = A_inv @ u
        w = v @ A_inv

        denom = a_kk - v @ s
        if abs(denom) < tol:
            raise ValueError(f"Matrix is singular at step {k+1}, denom = {denom:.3e}")

        x_k1 = (b_k1 - v @ x) / denom

        x_new = np.empty(k + 1)
        x_new[:k] = x - s * x_k1
        x_new[k] = x_k1
        x = x_new

        det = det * denom

        top_left = A_inv + np.outer(s, w) / denom
        top_right = -s / denom
        bottom_left = -w / denom
        bottom_right = 1.0 / denom

        A_inv_new = np.empty((k + 1, k + 1))
        A_inv_new[:k, :k] = top_left
        A_inv_new[:k, k] = top_right
        A_inv_new[k, :k] = bottom_left
        A_inv_new[k, k] = bottom_right
        A_inv = A_inv_new

        if verbose:
            print(f"\nШаг {k+1} (окаймление до размера {k+1}×{k+1}):")
            print(f"  Новый столбец u = {u}")
            print(f"  Новая строка   v = {v}")
            print(f"  Нижний правый a_kk = {a_kk:.6f}")
            print(f"  Дополнение a_k = a_kk - v·A⁻¹·u = {denom:.6f}")
            print(f"  Новая компонента решения x_{k+1} = {x_k1:.6f}")
            print(f"  Обновлённое решение x = {x}")
            print(f"  Текущий определитель = {det:.6f}")

    return x, det

if __name__ == "__main__":

    np.random.seed(67)
    n = 7

    while True:
        A_int = np.random.randint(-8, 9, size=(n, n))
        det_np = np.linalg.det(A_int.astype(float))
        if abs(det_np) > 1e-8:
            break

    A = A_int.astype(float)
    x_true = np.random.randint(-5, 6, size=n).astype(float)
    b = A @ x_true

    print(f"\n Решение системы размера {n}×{n} методом окаймления")
    x_sol, det_computed = solve_by_bordering_with_det(A, b, verbose=True)

    error_x = np.max(np.abs(x_sol - x_true))
    det_true = np.linalg.det(A)

    print(f"\nМатрица A:\n{A_int}")
    print(f"\nОпределитель: {det_computed:.6f}")

    print(f"\nИстинное решение: {x_true}")
    print(f"Найденное решение: {x_sol}")
    print(f"\nМаксимальная погрешность: {error_x:.10e}")


 Решение системы размера 7×7 методом окаймления
Шаг 1 (k=1): инициализация
  A_1 = [-5.0000]
  A_1⁻¹ = [-0.200000]
  x = [2.800000]
  det = -5.000000

Шаг 2 (окаймление до размера 2×2):
  Новый столбец u = [2.]
  Новая строка   v = [-5.]
  Нижний правый a_kk = 0.000000
  Дополнение a_k = a_kk - v·A⁻¹·u = -2.000000
  Новая компонента решения x_2 = 18.500000
  Обновлённое решение x = [10.2 18.5]
  Текущий определитель = 10.000000

Шаг 3 (окаймление до размера 3×3):
  Новый столбец u = [-3. -5.]
  Новая строка   v = [ 2. -1.]
  Нижний правый a_kk = -6.000000
  Дополнение a_k = a_kk - v·A⁻¹·u = -7.000000
  Новая компонента решения x_3 = 0.271429
  Обновлённое решение x = [ 9.92857143 18.22857143  0.27142857]
  Текущий определитель = -70.000000

Шаг 4 (окаймление до размера 4×4):
  Новый столбец u = [-1. -4.  1.]
  Новая строка   v = [ 1. -4.  2.]
  Нижний правый a_kk = 4.000000
  Дополнение a_k = a_kk - v·A⁻¹·u = 9.842857
  Новая компонента решения x_4 = 2.584906
  Обновлённое решение x =

# Лаба 7



```
Метод простой итерации решения слау
```


In [ ]:
import numpy as np

def generate_nonsingular_matrix(n=4, max_val=10):
    while True:
        A = np.random.randint(-max_val, max_val + 1, size=(n, n))
        if abs(np.linalg.det(A)) > 1e-8:
            return A.astype(float)

def make_diagonally_dominant(A, b):
    n = A.shape[0]
    A_dd = A.copy()
    for i in range(n):
        row_sum = np.sum(np.abs(A_dd[i, :])) - np.abs(A_dd[i, i])
        A_dd[i, i] = (1 if A_dd[i, i] >= 0 else -1) * (row_sum + 1)
    return A_dd, b

def simple_iteration_method_verbose(A, b, eps=1e-6, max_iter=10000):
    n = A.shape[0]
    x_prev = np.zeros(n)
    print(f"Начальное приближение: x = {x_prev}")

    for k in range(1, max_iter + 1):
        x_curr = np.empty(n)
        for i in range(n):
            sum_ax = np.dot(A[i, :i], x_prev[:i]) + np.dot(A[i, i+1:], x_prev[i+1:])
            x_curr[i] = (b[i] - sum_ax) / A[i, i]

        diff = np.abs(x_curr - x_prev)
        max_diff = np.max(diff)

        print(f"\nИтерация {k}:")
        print(f"  x^{k} = {x_curr}")
        print(f"  |x^{k} - x^{k-1}| = {diff}")
        print(f"  max |x^{k} - x^{k-1}| = {max_diff:.2e}")

        if max_diff <= eps:
            residual = np.linalg.norm(A @ x_curr - b, ord=np.inf)
            return x_curr, residual, k

        x_prev = x_curr.copy()

    raise RuntimeError("Метод не сошёлся за максимальное число итераций")

if __name__ == "__main__":
    np.random.seed(42)

    n = 8

    A = generate_nonsingular_matrix(n=n)
    b = np.random.randint(-10, 10, size=n).astype(float)
    A_dd, b_dd = make_diagonally_dominant(A, b)

    eps = 1e-12

    print(f"Матрица A (диагонально доминирующая) размера {n}×{n}:")
    print(A_dd)
    print("\nВектор правой части b:")
    print(b_dd)
    print(f"\nТочность (eps): {eps}")

    try:
        solution, error, iterations = simple_iteration_method_verbose(A_dd, b_dd, eps=eps)
        print(f"\nРешение x = {solution}")
        print(f"Невязка ||Ax - b|| = {error:.2e}")
        print(f"Общее количество итераций: {iterations}")
    except RuntimeError as e:
        print("Ошибка:", e)

Матрица A (диагонально доминирующая) размера 8×8:
[[-39.   9.   4.   0.  -3.  10.  -4.   8.]
 [  0.  48.  10.  -7.  -3.  -8.  10.  -9.]
 [  1.  -5. -35.  10. -10.   1.   1.   6.]
 [ -1.   5.   4.  37.   8.   1.   9.  -8.]
 [ -6.   8.  -4.  10. -47.  -4.   7.  -7.]
 [  3.   7.  -2.  10.  -9.  40.   4.  -4.]
 [  1.  -3.   4.  -8.   3.   6. -33.   7.]
 [ -3.  -7.  -9.  -5.  -1.  -7.   7.  40.]]

Вектор правой части b:
[-9. -1. -7.  3.  5.  4. -3.  3.]

Точность (eps): 1e-12
Начальное приближение: x = [0. 0. 0. 0. 0. 0. 0. 0.]

Итерация 1:
  x^1 = [ 0.23076923 -0.02083333  0.2         0.08108108 -0.10638298  0.1
  0.09090909  0.075     ]
  |x^1 - x^0| = [0.23076923 0.02083333 0.2        0.08108108 0.10638298 0.1
 0.09090909 0.075     ]
  max |x^1 - x^0| = 2.31e-01

Итерация 2:
  x^2 = [ 0.2863593  -0.04553484  0.28144245  0.082914   -0.14530016  0.05054079
  0.12880217  0.14272833]
  |x^2 - x^1| = [0.05559007 0.02470151 0.08144245 0.00183292 0.03891718 0.04945921
 0.03789308 0.06772833]
  

# Лаба 8



```
Степенной метод
```


In [ ]:
import numpy as np

def power_method(A, x0=None, tol=1e-14, max_iter=10000, verbose=True):

    n = A.shape[0]
    if x0 is None:
        x = np.random.rand(n)
    else:
        x = np.array(x0, dtype=float)

    # Нормируем
    x = x / np.linalg.norm(x, ord=np.inf)

    lam_old = 0.0

    if verbose:
        print(f"{'Итерация':>8} | {'λ (текущее)':>15} | {'||A·x - λ·x||_∞':>18}")
        print("-" * 52)

    for k in range(1, max_iter + 1):
        y = A @ x

    # Оценка λ через компоненту с наибольшим |x_i| (т.к. ||x||_∞ = 1 → |x_i| = 1 для этого i)
        idx = np.argmax(np.abs(x))
        lam = y[idx] / x[idx]

    # Нормируем y → новый x
        x_new = y / np.linalg.norm(y, ord=np.inf)

    # Невязка
        residual = np.linalg.norm(A @ x_new - lam * x_new, ord=np.inf)

        if verbose:
            print(f"{k:8d} | {lam:15.15f} | {residual:18.2e}")

    # Проверка сходимости по λ
        if np.abs(lam - lam_old) < tol * (np.abs(lam) + 1e-16):
            return lam, x_new, k

        x = x_new
        lam_old = lam

    raise RuntimeError("Степенной метод не сошёлся за заданное число итераций.")



if __name__ == "__main__":
    np.set_printoptions(precision=6, suppress=True)

    n = 7
    np.random.seed(42)

    A = np.random.randint(-5, 6, size=(n, n))


    for i in range(n):
        A[i, i] += 10

    print("Сгенерированная целочисленная матрица A:")
    print(A, "\n")

    lam, vec, iters = power_method(A, tol=1e-14, verbose=True)

    print(f"\nДоминирующее собственное значение: λ ≈ {lam:.12f}")
    print(f"\nСобственный вектор (нормирован по ∞-норме):")
    print(f"x = {vec}")
    print(f"\nЧисло итераций: {iters}")


    Ax = A @ vec
    lambda_x = lam * vec
    residual = np.linalg.norm(Ax - lambda_x, ord=np.inf)

    print(f"\nНевязка: ||A·x − λ·x||_∞ = {residual:.2e}")



Сгенерированная целочисленная матрица A:
[[11 -2  5  2 -1  1  4]
 [-3 11  5  5  2 -1 -2]
 [ 2  2  7  0 -1 -4  2]
 [ 0 -4 -1  5  4  0  3]
 [-5  5  5  4  7  1 -2]
 [ 3 -3 -1 -3  1  9  3]
 [ 1 -4 -2  3 -4  4 13]] 

Итерация |     λ (текущее) |    ||A·x - λ·x||_∞
----------------------------------------------------
       1 | 6.833030304229730 |           4.65e+00
       2 | 10.235437978180251 |           3.07e+00
       3 | 11.151291799848988 |           5.55e+00
       4 | 13.177894309890370 |           7.22e+00
       5 | 14.129845033141134 |           8.21e+00
       6 | 15.445736078131045 |           5.59e+00
       7 | 21.039169754614019 |           1.72e+00
       8 | 20.294654769487515 |           6.85e-01
       9 | 20.014826121911227 |           4.81e-01
      10 | 19.936386272662219 |           3.16e-01
      11 | 19.940609864524319 |           1.89e-01
      12 | 19.971458918517342 |           1.03e-01
      13 | 20.004252286350169 |           4.94e-02
      14 | 20.03016980416

In [ ]:
import numpy as np

def power_method(A, x0=None, tol=1e-14, max_iter=10000, verbose=True):
    n = A.shape[0]
    if x0 is None:
        x = np.random.rand(n)
    else:
        x = np.array(x0, dtype=float)

    # Нормируем по ∞-норме
    x = x / np.linalg.norm(x, ord=np.inf)

    lam_old = 0.0

    if verbose:
        print(f"{'Итерация':>8} | {'λ (текущее)':>15} | {'||A·x - λ·x||_∞':>18}")
        print("-" * 52)

    for k in range(1, max_iter + 1):
        y = A @ x

        # Оценка λ через компоненту с наибольшим |x_i|
        idx = np.argmax(np.abs(x))
        lam = y[idx] / x[idx]

        # Нормируем y → новый x
        x_new = y / np.linalg.norm(y, ord=np.inf)

        # Невязка
        residual = np.linalg.norm(A @ x_new - lam * x_new, ord=np.inf)

        if verbose:
            print(f"{k:8d} | {lam:15.15f} | {residual:18.2e}")

        # Проверка сходимости по λ
        if np.abs(lam - lam_old) < tol * (np.abs(lam) + 1e-16):
            return lam, x_new, k

        x = x_new
        lam_old = lam

    raise RuntimeError("Степенной метод не сошёлся за заданное число итераций.")


if __name__ == "__main__":
    np.set_printoptions(precision=6, suppress=True)

    # Заданная матрица A
    A = np.array([
        [-0.168700,  0.353699,  0.008540,  0.733624],
        [ 0.353699,  0.056519, -0.723182, -0.076440],
        [ 0.008540, -0.723182,  0.015938,  0.342333],
        [ 0.733624, -0.076440,  0.342333, -0.045744],
    ])

    print("Заданная матрица A:")
    print(A, "\n")

    lam, vec, iters = power_method(A, tol=1e-14, verbose=True)

    print(f"\nДоминирующее собственное значение: λ ≈ {lam:.12f}")
    print(f"\nСобственный вектор (нормирован по ∞-норме):")
    print(f"x = {vec}")
    print(f"\nЧисло итераций: {iters}")

    # Проверка невязки
    Ax = A @ vec
    lambda_x = lam * vec
    residual = np.linalg.norm(Ax - lambda_x, ord=np.inf)

    print(f"\nНевязка: ||A·x − λ·x||_∞ = {residual:.2e}")

Заданная матрица A:
[[-0.1687    0.353699  0.00854   0.733624]
 [ 0.353699  0.056519 -0.723182 -0.07644 ]
 [ 0.00854  -0.723182  0.015938  0.342333]
 [ 0.733624 -0.07644   0.342333 -0.045744]] 

Итерация |     λ (текущее) |    ||A·x - λ·x||_∞
----------------------------------------------------
       1 | 0.038569133783961 |           9.54e-01
       2 | 0.491838961762715 |           6.64e-01
       3 | 0.154352358253307 |           1.00e+00
       4 | 0.107411113995324 |           6.83e-01
       5 | 0.263574736523597 |           8.69e-01
       6 | 0.228812154006542 |           6.92e-01
       7 | 0.362733407267803 |           7.41e-01
       8 | 0.313048997189259 |           6.86e-01
       9 | 0.449381457778037 |           6.21e-01
      10 | 0.367356248774353 |           6.83e-01
      11 | 0.522274437401768 |           5.15e-01
      12 | 0.396391249733764 |           6.93e-01
      13 | 0.581248193218961 |           5.06e-01
      14 | 0.403582477683366 |           7.22e-01
    

In [ ]:
import numpy as np

def power_method(A, x0=None, tol=1e-14, max_iter=10000, verbose=True):
    n = A.shape[0]
    if x0 is None:
        x = np.random.rand(n)
    else:
        x = np.array(x0, dtype=float)

    # Нормируем по ∞-норме
    x = x / np.linalg.norm(x, ord=np.inf)

    lam_old = 0.0

    if verbose:
        print(f"{'Итерация':>8} | {'λ (текущее)':>15} | {'||A·x - λ·x||_∞':>18}")
        print("-" * 52)

    for k in range(1, max_iter + 1):
        y = A @ x

        # Оценка λ через компоненту с наибольшим |x_i|
        idx = np.argmax(np.abs(x))
        lam = y[idx] / x[idx]

        # Нормируем y → новый x
        x_new = y / np.linalg.norm(y, ord=np.inf)

        # Невязка
        residual = np.linalg.norm(A @ x_new - lam * x_new, ord=np.inf)

        if verbose:
            print(f"{k:8d} | {lam:15.15f} | {residual:18.2e}")

        # Проверка сходимости по λ
        if np.abs(lam - lam_old) < tol * (np.abs(lam) + 1e-16):
            return lam, x_new, k

        x = x_new
        lam_old = lam

    raise RuntimeError("Степенной метод не сошёлся за заданное число итераций.")


if __name__ == "__main__":
    np.set_printoptions(precision=6, suppress=True)

    # Заданная матрица A
    A = np.array([
        [2.2,  1,  0.5,  2],
        [ 1,  1.3, 2, 1],
        [ 0.5, 2,  0.5,  1.6],
        [2, 1,  1.6, 2],
    ])

    print("Заданная матрица A:")
    print(A, "\n")

    lam, vec, iters = power_method(A, tol=1e-14, verbose=True)

    print(f"\nДоминирующее собственное значение: λ ≈ {lam:.12f}")
    print(f"\nСобственный вектор (нормирован по ∞-норме):")
    print(f"x = {vec}")
    print(f"\nЧисло итераций: {iters}")

    # Проверка невязки
    Ax = A @ vec
    lambda_x = lam * vec
    residual = np.linalg.norm(Ax - lambda_x, ord=np.inf)

    print(f"\nНевязка: ||A·x − λ·x||_∞ = {residual:.2e}")

Заданная матрица A:
[[2.2 1.  0.5 2. ]
 [1.  1.3 2.  1. ]
 [0.5 2.  0.5 1.6]
 [2.  1.  1.6 2. ]] 

Итерация |     λ (текущее) |    ||A·x - λ·x||_∞
----------------------------------------------------
       1 | 3.227239233882978 |           2.98e+00
       2 | 5.351361270572257 |           5.01e-01
       3 | 5.512404317394353 |           1.77e-01
       4 | 5.689713117662452 |           4.64e-02
       5 | 5.643272578116167 |           1.11e-02
       6 | 5.654420390077737 |           2.94e-03
       7 | 5.651483076377756 |           7.01e-04
       8 | 5.652184166403957 |           1.86e-04
       9 | 5.651997962932944 |           4.40e-05
      10 | 5.652042000360740 |           1.18e-05
      11 | 5.652030185041271 |           2.76e-06
      12 | 5.652032948385632 |           7.50e-07
      13 | 5.652032197959743 |           1.73e-07
      14 | 5.652032371158879 |           4.77e-08
      15 | 5.652032323445859 |           1.08e-08
      16 | 5.652032334286480 |           3.04e-09


In [ ]:
import numpy as np

def power_method(A, x0=None, tol=1e-14, max_iter=10000, verbose=True):
    n = A.shape[0]
    if x0 is None:
        x = np.random.rand(n)
    else:
        x = np.array(x0, dtype=float)

    # Нормируем по ∞-норме
    x = x / np.linalg.norm(x, ord=np.inf)

    lam_old = 0.0

    if verbose:
        print(f"{'Итерация':>8} | {'λ (текущее)':>15} | {'||A·x - λ·x||_∞':>18}")
        print("-" * 52)

    for k in range(1, max_iter + 1):
        y = A @ x

        # Оценка λ через компоненту с наибольшим |x_i|
        idx = np.argmax(np.abs(x))
        lam = y[idx] / x[idx]

        # Нормируем y → новый x
        x_new = y / np.linalg.norm(y, ord=np.inf)

        # Невязка
        residual = np.linalg.norm(A @ x_new - lam * x_new, ord=np.inf)

        if verbose:
            print(f"{k:8d} | {lam:15.15f} | {residual:18.2e}")

        # Проверка сходимости по λ
        if np.abs(lam - lam_old) < tol * (np.abs(lam) + 1e-16):
            return lam, x_new, k

        x = x_new
        lam_old = lam

    raise RuntimeError("Степенной метод не сошёлся за заданное число итераций.")


if __name__ == "__main__":
    np.set_printoptions(precision=6, suppress=True)

    # Матрица A из вашего изображения
    A = np.array([
        [1.00, 0.42, 0.54, 0.66],
        [0.42, 1.00, 0.32, 0.44],
        [0.54, 0.32, 1.00, 0.22],
        [0.66, 0.44, 0.22, 1.00],
    ])

    print("Матрица A из изображения:")
    print(A, "\n")

    lam, vec, iters = power_method(A, tol=1e-14, verbose=True)

    print(f"\nДоминирующее собственное значение: λ ≈ {lam:.12f}")
    print(f"\nСобственный вектор (нормирован по ∞-норме):")
    print(f"x = {vec}")
    print(f"\nЧисло итераций: {iters}")

    # Проверка невязки
    Ax = A @ vec
    lambda_x = lam * vec
    residual = np.linalg.norm(Ax - lambda_x, ord=np.inf)

    print(f"\nНевязка: ||A·x − λ·x||_∞ = {residual:.2e}")

Матрица A из изображения:
[[1.   0.42 0.54 0.66]
 [0.42 1.   0.32 0.44]
 [0.54 0.32 1.   0.22]
 [0.66 0.44 0.22 1.  ]] 

Итерация |     λ (текущее) |    ||A·x - λ·x||_∞
----------------------------------------------------
       1 | 1.571208204191998 |           8.17e-01
       2 | 2.388174939551112 |           1.18e-01
       3 | 2.323976269943103 |           2.24e-02
       4 | 2.321210857975222 |           6.08e-03
       5 | 2.322068358702727 |           2.12e-03
       6 | 2.322513674938415 |           7.37e-04
       7 | 2.322671887323198 |           2.51e-04
       8 | 2.322723884981056 |           8.54e-05
       9 | 2.322740702891980 |           2.90e-05
      10 | 2.322746150819990 |           9.87e-06
      11 | 2.322747927072704 |           3.38e-06
      12 | 2.322748510493508 |           1.17e-06
      13 | 2.322748703461332 |           4.02e-07
      14 | 2.322748767680926 |           1.38e-07
      15 | 2.322748789166857 |           4.77e-08
      16 | 2.322748796387712